# Attribute 7 - Buffer contribution: shape optimisation
#### Rationale
Add to protection of an existing natural area (ENA), improves it shape and/or increase its size. Higher score for proximity to/ buffer for a priority management site .

#### Data Layers
Proximity to ENA (maps of realms created by Eco-index)/ shape optimisation model (using Convex Hulls in GIS – David to generate internally)
DOC EMU link 
QEII link (need to ask for permission to use – also in land status category) 

#### Scoring
1 if improves shape of an existing patch, else 0


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import math
import os
from os import listdir

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from constants import small_polygon_threshold, m2_to_ha, x_resolution, y_resolution, keep_cols, keep_cols_catch

# Load

In [3]:
%%time
# Area of Interest
aoi = gpd.read_file("../BaseLayersEco-index/Eco-index_RestorableAreas__Catchments_v290824.gpkg")
aoi = aoi[['Catchment', 'geometry']].copy()
aoi.sindex
aoi.shape

CPU times: total: 1.36 s
Wall time: 2.75 s


(45989, 2)

In [4]:
%%time
## This gets us the existing native vegetation
lcs = gpd.read_file('../BaseLayersEco-index/Eco-index_LandCoverSnapshot_v290824.gpkg')
lcs.sindex
lcs.shape

CPU times: total: 9.53 s
Wall time: 19.5 s


(511090, 6)

In [5]:
%%time
tot = pd.read_csv('../BaseLayersEco-index/TableOfTruth_LCDB_Mapping.csv')

tot = tot[['LCDB_5 Land cover class', 'LCDB Wetland context', 'Land cover status',
       'Existing Natural Area', 'Eco-Index Realm (Generalised, not IUCN)','LCDB Class "Description" for heat map attributes #3  and #7 - use Native Veg and also Wetland with exotic canopy']]
# # tot = tot[['LCDB_5 Land cover class', 'LCDB Wetland context', 'Land cover status']].copy()
    
anys = tot.loc[tot['LCDB Wetland context'] == 'any', :].copy()
anys['LCDB Wetland context'] = anys.shape[0]*[['Yes', 'No']]

tot_fixed = pd.concat([
    anys.explode('LCDB Wetland context'), # the ones we have added
    tot[tot['LCDB Wetland context'] != 'any']
]).reset_index(drop=True)

# # add cols for easier merging
tot_fixed['Name_2018'] = tot_fixed['LCDB_5 Land cover class'].copy()
tot_fixed['Wetland_18'] = tot_fixed['LCDB Wetland context'].str.lower()
tot_fixed['Realm'] = tot_fixed['Eco-Index Realm (Generalised, not IUCN)'].copy()
tot_fixed['Indig'] = tot_fixed['LCDB Class "Description" for heat map attributes #3  and #7 - use Native Veg and also Wetland with exotic canopy'].copy()
# tot_fixed = tot_fixed.drop(['LCDB Wetland context', 
#                             'LCDB_5 Land cover class', 
#                             'Eco-Index Realm (Generalised, not IUCN)'
#                            'LCDB Class "Description" for heat map attributes #3  and #7'], axis=1)

tot_fixed_indig = tot_fixed[(~pd.isna(tot_fixed['Indig'])) | 
                            ((tot_fixed.Name_2018=='Deciduous Hardwoods') & (tot_fixed['LCDB Wetland context']=='Yes'))].copy()

ena = lcs[lcs.Realm.isin(['Freshwater', 'Terrestrial'])].copy()
ena = ena.merge(tot_fixed_indig[['Name_2018', 'Wetland_18', 'Indig']], how='inner', left_on=['LCDBLandCoverClass', 'LCDBWetlandContext'], right_on=['Name_2018', 'Wetland_18'])


CPU times: total: 266 ms
Wall time: 282 ms


In [6]:
%%time
ana = ena.copy()
del ena
ana = ana[['geometry']].dissolve()
ana.sindex
ana = ana.explode(index_parts=False) #explode to make sure we're doing convex hull on seperate polygons
ana.sindex

CPU times: total: 7min 8s
Wall time: 7min 9s


In [7]:
ana.to_file(f"../OutputArtifacts/A07_NativeVegetationShapeImprovement/A07_IntermediateLayers/ana_dissolved_explode_20240906.gpkg")

In [8]:
%%time
ha_threshold = 100

convex = ana[ana.area * m2_to_ha < ha_threshold].copy() # Remove very large areas
convex['geometry'] = convex.convex_hull

CPU times: total: 1.8 s
Wall time: 1.79 s


In [9]:
convex_aoi = convex.overlay(aoi)

C:\Users\dav\miniconda3_9\envs\eco\Lib\site-packages\geopandas\geodataframe.py:2675: UserWarning: `keep_geom_type=True` in overlay resulted in 568032 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  return geopandas.overlay(


In [10]:
from constants import small_polygon_threshold

convex_aoi = convex_aoi.explode()
convex_aoi["PrioOption"] = "Native Vegetation Shape Improvement"
convex_aoi["PixelScore"] = 6
convex_aoi["PixelDesc"] = "Improves the shape of an existing natural area"
convex_aoi["Area_ha"] = convex_aoi.area * m2_to_ha

convex_aoi = convex_aoi[convex_aoi.Area_ha > small_polygon_threshold].reset_index(drop=False)
convex_aoi["Area_ha"] = convex_aoi["Area_ha"].round(2)

convex_aoi[keep_cols_catch].to_file(
    f"../OutputArtifacts/A07_NativeVegetationShapeImprovement/A07_NativeVegetationShapeImprovement_20240906.gpkg"
)

In [11]:
convex_aoi.head()

,index,Catchment,geometry,PrioOption,PixelScore,PixelDesc,Area_ha
0,0,Waiau-sth,"POLYGON ((1182967.296 4871599.683, 1182976.974...",Native Vegetation Shape Improvement,6,Improves the shape of an existing natural area,4.94
1,0,Waiau-sth,"POLYGON ((1182871.32 4871557.198, 1183145.65 4...",Native Vegetation Shape Improvement,6,Improves the shape of an existing natural area,1.20
2,0,Waiau-sth,"POLYGON ((1183637.234 4872364.606, 1183617.525...",Native Vegetation Shape Improvement,6,Improves the shape of an existing natural area,4.53
3,0,Waiau-sth,"POLYGON ((1183309.919 4872546.351, 1183276.03 ...",Native Vegetation Shape Improvement,6,Improves the shape of an existing natural area,0.10
4,0,Waiau-sth,"POLYGON ((1183669.443 4872780.862, 1183623.446...",Native Vegetation Shape Improvement,6,Improves the shape of an existing natural area,0.24


In [12]:
import time
for n_catch, catchment in enumerate(aoi.Catchment.sort_values().unique()):
    current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(time.time()))
    print(f'\n{catchment.upper()}\n{current_time}')
    
    sub_convex_aoi = convex_aoi[convex_aoi['Catchment'] == catchment]
    
    if sub_convex_aoi.shape[0] ==0:
        continue
        
    sub_convex_aoi[keep_cols_catch].to_file(
        f"../OutputArtifacts/A07_NativeVegetationShapeImprovement/A07_Catchments/{str(n_catch).zfill(3)}_{catchment}_ShapeImprovement_20240906.gpkg"
    )


APARIMA
2024-09-16 16:33:16

ASHBURTON-HINDS
2024-09-16 16:33:16

AUCKLAND BASIN
2024-09-16 16:33:16

AUCKLAND OFFSHORE ISLANDS
2024-09-16 16:33:16

AWATERE
2024-09-16 16:33:16

BANKS PENINSULA
2024-09-16 16:33:16

BAY OF PLENTY OFFSHORE ISLANDS
2024-09-16 16:33:17

BLUFF OFFSHORE
2024-09-16 16:33:17

BULLER
2024-09-16 16:33:17

CATLINS
2024-09-16 16:33:17

CLARENCE
2024-09-16 16:33:17

CLUTHA
2024-09-16 16:33:17

CONWAY
2024-09-16 16:33:17

COROMANDEL
2024-09-16 16:33:17

COROMANDEL OFFSHORE ISLANDS
2024-09-16 16:33:17

EAST-CAPE
2024-09-16 16:33:18

FAR NORTH
2024-09-16 16:33:18

FAR NORTH OFFSHORE ISLANDS
2024-09-16 16:33:18

FIORDLAND
2024-09-16 16:33:18

FOX-WHATAROA
2024-09-16 16:33:18

GOLDEN BAY
2024-09-16 16:33:18

GREAT BARRIER ISLANDS
2024-09-16 16:33:18

GREY
2024-09-16 16:33:18

HAAST
2024-09-16 16:33:18

HAURAKI
2024-09-16 16:33:18

HAWKES BAY
2024-09-16 16:33:19

HOKIANGA HARBOUR
2024-09-16 16:33:19

HOKITIKA
2024-09-16 16:33:19

HURUNUI
2024-09-16 16:33:19

HUTT
2024-0